In [ ]:
import matplotlib.pyplot as plt
from mxlpy import Simulator, make_protocol
from mxlmodels import get_lam2026
import pandas as pd

from mxlpy import Model
import numpy as np

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
protocol_5_10_5 = make_protocol([
    (1/60, {"ppfd": 0}),
    (5, {"ppfd": 1}),
    (10, {"ppfd": 0}),
    (5, {"ppfd": 1}),
])

protocol_baseline = make_protocol([
    (1/60, {"ppfd": 0}),
    (20, {"ppfd": 1}),
])

protocol_3_1_1_3_9_3 = make_protocol([
    (1/60, {"ppfd": 0}),
    (3, {"ppfd": 1}),
    (1, {"ppfd": 0}),
    (1, {"ppfd": 1}),
    (3, {"ppfd": 0}),
    (9, {"ppfd": 1}),
    (3, {"ppfd": 0}),
])

In [ ]:
m_WT = get_lam2026()
s_WT = Simulator(m_WT).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_WT = s_WT.get_result().unwrap_or_err().get_combined()
m_WT = get_lam2026()
s_WT_baseline = Simulator(m_WT).simulate_protocol(protocol_baseline, time_points_per_step=100)
res_WT_baseline = s_WT_baseline.get_result().unwrap_or_err().get_combined()
m_WT = get_lam2026()
s_WT_311393 = Simulator(m_WT).simulate_protocol(protocol_3_1_1_3_9_3, time_points_per_step=100)
res_WT_311393 = s_WT_311393.get_result().unwrap_or_err().get_combined()


In [ ]:
m_npq4npq1 = get_lam2026().update_parameters(
    {
        "tau_0": 1.283586052,
        "kappa_QV": 0,
        "kappa_QA": 0,
        "kappa_QZ": 0,
        "kappa_QL": 0,
        "kappa_qZ": 0,
        "kappa_qI": 7.05,
    }
)
s_npq4npq1 = Simulator(m_npq4npq1).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_npq4npq1 = s_npq4npq1.get_result().unwrap_or_err().get_combined()

In [ ]:
m_npq1 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.693954731,
        "kappa_qZ": 0,
        "kappa_QA": 0,
        "kappa_QZ": 0,
        "V_tot_WT": 49.8,
    })
    .remove_reaction("VA")
    .remove_reaction("AV")
    .remove_reaction("AZ")
    .remove_reaction("ZA")
    .remove_reaction("PAf")
    .remove_reaction("PAb")
    .remove_reaction("PZf")
    .remove_reaction("PZb")
    .remove_reaction("QAb")
    .remove_reaction("QAf")
    .remove_reaction("QZf")
    .remove_reaction("QZb")
)
s_npq1 = Simulator(m_npq1).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_npq1 = s_npq1.get_result().unwrap_or_err().get_combined()


In [ ]:
m_npq4 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.702050733,
        "V_tot_WT": 40.6,
        "kappa_QV": 0,
        "kappa_QA": 0,
        # "kappa_QZ": 0,
        "kappa_QL": 0,
    })
    .remove_reaction("QVf")
    .remove_reaction("QVb")
    .remove_reaction("QAb")
    .remove_reaction("QAf")
    .remove_reaction("QZf")
    .remove_reaction("QZb")
)
s_npq4 = Simulator(m_npq4).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_npq4 = s_npq4.get_result().unwrap_or_err().get_combined()


In [ ]:
m_lut2 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.453352343,
        "V_tot_WT": 71.2,
        "kappa_QL": 0,
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "P_tot": 49.9,
    })
    .update_variables(
        {
            "PL": 0,
         }
    )
)
s_lut2 = Simulator(m_lut2).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_lut2 = s_lut2.get_result().unwrap_or_err().get_combined()


In [ ]:
m_zep2 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.453352343,
        "V_tot_WT": 10.7,
        # "X_tot": 87.5,
        "k_AV": 0.006,
        "k_ZA": 0.038,
    })
    .update_variables(
        {   
            # "V": 33.4,
            # "A": 23.4,
            # "Z": 30.6,
            "PL": 152.7,
         }
    )
)
s_zep2 = Simulator(m_zep2).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_zep2 = s_zep2.get_result().unwrap_or_err().get_combined()


In [ ]:
from matplotlib.patches import Rectangle

fig, axs = plt.subplots(ncols = 3, nrows = 2, figsize = (12, 6), sharey= True, sharex= True)

res_list = [res_npq4npq1, res_npq4, res_npq1, res_lut2, res_zep2]
label_list = ["npq4npq1", "npq4", "npq1", "lut2","zep2"]

for ax, res, label in zip(axs.flatten(), res_list, label_list):
    ax.plot(res.index, res["tau_Fluo"], label = label, color = "red")
    ax.set_ylim(0,1.9)
    ax.set_ylabel(r"$\tau_{avg}$")
    ax.set_xlabel("Time (min)")
    ax.set_title(label)

for ax in axs.flatten():
    ax.add_patch(Rectangle(
        (5, 0.01), 10, 0.05,       # x, y, width, height in data coords
        transform=ax.get_xaxis_transform(),  # x: data, y: axes fraction
        clip_on=False, facecolor="black", zorder=5, edgecolor ='black'))
    ax.add_patch(Rectangle(
        (0, 0.01), 5, 0.05,       # x, y, width, height in data coords
        transform=ax.get_xaxis_transform(),  # x: data, y: axes fraction
        clip_on=False, facecolor="white", zorder=5, edgecolor ='black'))
    ax.add_patch(Rectangle(
        (15, 0.01), 5, 0.05,       # x, y, width, height in data coords
        transform=ax.get_xaxis_transform(),  # x: data, y: axes fraction
        clip_on=False, facecolor="white", zorder=5, edgecolor ='black'))

fig.tight_layout()

In [ ]:
m_npq1lut2 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.453352343,
        "V_tot_WT": 49.8,#71.2, #
        # "X_tot": 49.8+1,#71.2+1,#
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "kappa_qI": 7.04,
    })
    .remove_reaction("VA")
    .remove_reaction("AV")
    .remove_reaction("AZ")
    .remove_reaction("ZA")
    .remove_reaction("PAf")
    .remove_reaction("PAb")
    .remove_reaction("PZf")
    .remove_reaction("PZb")
    .remove_reaction("QAb")
    .remove_reaction("QAf")
    .remove_reaction("QZf")
    .remove_reaction("QZb")
    .update_variables(
        {
            "PL": 49.9,
         }
    )
)
s_npq1lut2 = Simulator(m_npq1lut2).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_npq1lut2 = s_npq1lut2.get_result().unwrap_or_err().get_combined()


In [ ]:
m_zep2lut2 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.453352343,
        "V_tot_WT": 10.7,
        # "X_tot": 11,
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "kappa_qI": 7.04,
    })
    .update_variables(
        {
            "PL": 49.9,
         }
    )
)
s_zep2lut2 = Simulator(m_zep2lut2).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_zep2lut2 = s_zep2lut2.get_result().unwrap_or_err().get_combined()


In [ ]:
m_npq4lut2 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.453352343,
        "V_tot_WT": 40.6,
        # "X_tot": 41.6,
        "kappa_QV": 0,
        "kappa_QA": 0,
        "kappa_QL": 0,
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "kappa_qI": 7.04,
    })
    .remove_reaction("QVf")
    .remove_reaction("QVb")
    .remove_reaction("QAb")
    .remove_reaction("QAf")
    .remove_reaction("QZf")
    .remove_reaction("QZb")
    .update_variables(
        {
            "PL": 49.9,
         }
    )
)
s_npq4lut2 = Simulator(m_npq4lut2).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_npq4lut2 = s_npq4lut2.get_result().unwrap_or_err().get_combined()


In [ ]:
m_npq4zep2 = (
    get_lam2026()
    .update_parameters(
    {
        "tau_0": 1.453352343,
        "V_tot_WT": 10.7,
        # "X_tot": 11,
        "kappa_QV": 0,
        "kappa_QA": 0,
        "kappa_QL": 0,
        "kappa_qI": 7.04,
    })
    .remove_reaction("QVf")
    .remove_reaction("QVb")
    .remove_reaction("QAb")
    .remove_reaction("QAf")
    .remove_reaction("QZf")
    .remove_reaction("QZb")
)
s_npq4zep2 = Simulator(m_npq4zep2).simulate_protocol(protocol_5_10_5, time_points_per_step=100)
res_npq4zep2 = s_npq4zep2.get_result().unwrap_or_err().get_combined()


In [ ]:
from matplotlib.patches import Rectangle

fig, axs = plt.subplots(ncols = 4, nrows = 2, figsize = (16, 6), sharey= True, sharex= True)

res_list = [res_WT, res_WT_baseline, res_WT_311393, res_zep2lut2, res_npq1lut2, res_npq4lut2, res_npq4zep2]
label_list = ["WT", "WT", "WT", "zep2lut2", "npq1lut2", "npq4lut2", "npq4zep2"]

for ax, res, label in zip(axs.flatten(), res_list, label_list):
    ax.plot(res.index, res["tau_Fluo"], label = label, color = "red")
    ax.set_ylim(0,1.9)
    ax.set_ylabel(r"$\tau_{avg}$")
    ax.set_xlabel("Time (min)")
    ax.set_title(label)

for ax in axs.flatten():
    ax.add_patch(Rectangle(
        (5, 0.01), 10, 0.05,       # x, y, width, height in data coords
        transform=ax.get_xaxis_transform(),  # x: data, y: axes fraction
        clip_on=False, facecolor="black", zorder=5, edgecolor ='black'))
    ax.add_patch(Rectangle(
        (0, 0.01), 5, 0.05,       # x, y, width, height in data coords
        transform=ax.get_xaxis_transform(),  # x: data, y: axes fraction
        clip_on=False, facecolor="white", zorder=5, edgecolor ='black'))
    ax.add_patch(Rectangle(
        (15, 0.01), 5, 0.05,       # x, y, width, height in data coords
        transform=ax.get_xaxis_transform(),  # x: data, y: axes fraction
        clip_on=False, facecolor="white", zorder=5, edgecolor ='black'))

fig.tight_layout()

In [ ]:
timeline = np.linspace(1,40,40)

con_WT = Simulator(m_WT).simulate(1e-6).update_parameter('ppfd',1).simulate_time_course(timeline).get_result().unwrap_or_err().get_combined()
con_npq1 = Simulator(m_npq1).simulate(1e-6).update_parameter('ppfd',1).simulate_time_course(timeline).get_result().unwrap_or_err().get_combined()
con_npq4 = Simulator(m_npq4).simulate(1e-6).update_parameter('ppfd',1).simulate_time_course(timeline).get_result().unwrap_or_err().get_combined()
con_zep2 = Simulator(m_zep2).simulate(1e-6).update_parameter('ppfd',1).simulate_time_course(timeline).get_result().unwrap_or_err().get_combined()
con_lut2 = Simulator(m_lut2).simulate(1e-6).update_parameter('ppfd',1).simulate_time_course(timeline).get_result().unwrap_or_err().get_combined()
con_npq4npq1 = Simulator(m_npq4npq1).simulate(1e-6).update_parameter('ppfd',1).simulate_time_course(timeline).get_result().unwrap_or_err().get_combined()

In [ ]:
res_list= [con_npq4npq1, con_npq4, con_npq1, con_lut2, con_zep2, con_WT]
timepoints = [1, 3, 5, 20]
# timepoints = [1, 6, 10, 40]
geno = ["npq4npq1", "npq4", "npq1", "lut2", "zep2", "WT"]
vars =  ["NPQ_qI", "NPQ_V", "NPQ_A", "NPQ_Z_qE", "NPQ_L", "NPQ_Z_qZ"]
colors = ["#7e2e8d", "#ecb11f", "#d85218", "#77ac2f", "#0072bd", "#8eec8e"]


fig, axs = plt.subplots(ncols= 2, nrows =2, figsize = (8, 8), sharey= True)

for ax, time in zip(axs.flatten(), timepoints):
    ax.set_ylim(0,6)
    dfs = []
    for res, gen in zip(res_list, geno):
        df = res.loc[[time], vars]
        df = df.T
        df.columns = [gen]
        dfs.append(df)
    dff = pd.concat(dfs, axis=1)
    height = 0
    bottom = 0
    for var, color in zip(vars, colors):
        # height = height + dff.loc[var]
        ax.bar(geno, dff.loc[var], bottom=bottom,label = var, color = color)
        bottom = bottom + dff.loc[var]
    ax.set_title(f"Time = {time} min")
    plt.legend()


In [ ]:
fig, axs = plt.subplots(nrows=2, ncols= 2, figsize = (8, 8), sharey= True )

tau_WT =  Simulator(get_lam2026()).simulate_protocol(protocol_5_10_5, time_points_per_step=100).get_result().unwrap_or_err().get_combined()["tau_Fluo"]
for ax in axs.flatten():
    ax.plot(tau_WT.index, tau_WT.values, label= f"WT")

for i in [2,5, 25]:
    m_plot = get_lam2026().update_parameters({
    "k_L_VA": 2.47*i,
    "k_D_VA": 0.014*i,
    "k_L_AZ": 0.5*i,
})
    res_tau = Simulator(m_plot).simulate_protocol(protocol_5_10_5, time_points_per_step=100).get_result().unwrap_or_err().get_combined()["tau_Fluo"]
    axs[0,0].plot(res_tau.index, res_tau.values, label= f"{i}X VDE")
    axs[0,0].set_ylim(0,2)
    axs[0,0].legend()

for i in [2,5, 25]:
    m_plot = get_lam2026().update_parameters({
    "k_AV": 2.47*i,
    "k_ZA": 0.014*i,
})
    res_tau = Simulator(m_plot).simulate_protocol(protocol_5_10_5, time_points_per_step=100).get_result().unwrap_or_err().get_combined()["tau_Fluo"]
    axs[0,1].plot(res_tau.index, res_tau.values, label= f"{i}X ZEP")
    axs[0,1].legend()

for i in [2,5, 25]:
    m_plot = get_lam2026().update_parameters({
    "kappa_QV": 0.040*i,
    "kappa_QA": 0.174*i,
    "kappa_QZ": 0.177*i,
    "kappa_QL": 0.262*i,
})
    res_tau = Simulator(m_plot).simulate_protocol(protocol_5_10_5, time_points_per_step=100).get_result().unwrap_or_err().get_combined()["tau_Fluo"]
    axs[1,0].plot(res_tau.index, res_tau.values, label= f"{i}X PsbS")
    axs[1,0].legend()

for i,j,k in zip([1,10], [5, 50], [2,5]):
    m_plot = get_lam2026().update_parameters({
    "k_L_VA": 2.47*i,
    "k_D_VA": 0.014*i,
    "k_L_AZ": 0.5*i,
    
    "k_AV": 2.47*j,
    "k_ZA": 0.014*j,

    "kappa_QV": 0.040*k,
    "kappa_QA": 0.174*k,
    "kappa_QZ": 0.177*k,
    "kappa_QL": 0.262*k,
})
    res_tau = Simulator(m_plot).simulate_protocol(protocol_5_10_5, time_points_per_step=100).get_result().unwrap_or_err().get_combined()["tau_Fluo"]
    axs[1,1].plot(res_tau.index, res_tau.values, label= f"{i}X VDE {j}X ZEP {k}X PsbS")
    axs[1,1].legend()


